# 07 — Preparação dos dados
Transformação de Fisher (z), vetorização do triângulo superior (190·189/2 = 17.955 atributos) e divisão treino/teste estratificada.
A padronização (`StandardScaler`) é ajustada **somente no treino**, dentro dos notebooks 08 e 09, para evitar vazamento.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import config

df = pd.read_csv(config.ARQ_PARTICIPANTES, dtype={"ID_padronizado": str})
fc = config.carregar_fc(df["ID_padronizado"])
y = df["classe"].values
ids = df["ID_padronizado"].values

In [ ]:
iu = np.triu_indices(config.N_ROIS, k=1)
fc_z = np.arctanh(np.clip(fc, -0.999999, 0.999999))   # Fisher z
X = fc_z[:, iu[0], iu[1]].astype(np.float32)
print("X:", X.shape, "| NaN:", np.isnan(X).sum())

In [ ]:
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, stratify=y, random_state=config.SEED)

print("Treino:", len(idx_tr), np.bincount(y[idx_tr]))
print("Teste: ", len(idx_te), np.bincount(y[idx_te]))
print("\nSites no teste:")
print(df.iloc[idx_te]["Site"].value_counts().sort_index())

In [ ]:
np.savez_compressed(
    config.ARQ_DATASET,
    X_train=X[idx_tr], y_train=y[idx_tr], ids_train=ids[idx_tr],
    X_test=X[idx_te],  y_test=y[idx_te],  ids_test=ids[idx_te],
)
print("Salvo:", config.ARQ_DATASET)